# COMP5318 Assignment 1: Rice Classification

##### Group number: 217
##### Student 1 SID: STUDENT_ID_REDACTED
##### Student 2 SID: STUDENT_ID_REDACTED 
##### Student 3 SID: STUDENT_ID_REDACTED 
##### Student 4 SID: STUDENT_ID_REDACTED

## **1. Data Pre-processing**

In [1]:
# Import all libraries
from pathlib import Path

import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
    GridSearchCV
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier
)
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [3]:
# Change only this filename to process another CSV with the same label format.
filename = "rice-final2.csv"
if not Path(filename).exists():
    filename = str(Path("clean_dataset") / filename)


In [4]:
def preprocess_data(filename):
    data = pd.read_csv(filename, na_values="?")

    # The last column is the label; all other columns are features.
    X = data.iloc[:, :-1]
    y = data.iloc[:, -1].map({"class1": 0, "class2": 1}).to_numpy()

    # Fill missing values with column means, then scale each feature to [0, 1].
    X = SimpleImputer(strategy="mean").fit_transform(X)
    X = MinMaxScaler().fit_transform(X)
    return X, y


X, y = preprocess_data(filename)


In [5]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec
# A function is provided to assist

def print_data(X, y, n_rows=10):
    """Takes a numpy data array and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: number of rows to print
    """
    for example_num in range(min(n_rows, len(X))):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])
            

print_data(X, y)


0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [6]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers

In [7]:
# Logistic Regression
logreg = LogisticRegression(solver="liblinear", random_state=0)
logreg_scores = cross_val_score(logreg, X, y, cv=cvKFold, scoring="accuracy")


In [8]:
# Naïve Bayes
naive_bayes = GaussianNB()
nb_scores = cross_val_score(naive_bayes, X, y, cv=cvKFold, scoring="accuracy")


### Part 1 Results


In [9]:
# Print results for each classifier in part 1 to 4 decimal places here:
print(f"LogR average cross-validation accuracy: {logreg_scores.mean():.4f}")
print(f"NB average cross-validation accuracy: {nb_scores.mean():.4f}")

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264


### Part 2: Cross-validation with parameter tuning

In [10]:
# KNN 
# parameters you may consider
knn_neighbors = [1, 3, 5, 7]
knn_distance_powers = [1, 2]

# Create the KNN classifier
knn = KNeighborsClassifier()

# Parameter grid for GridSearchCV
knn_param_grid = {
    "n_neighbors": knn_neighbors,
    "p": knn_distance_powers
}

In [11]:
# Decision Tree 
# parameters you may consider
dt_max_depth = [3, 5, 7, 10]
dt_min_samples_split = [2, 5, 10]
dt_min_samples_leaf = [1, 2, 4]

# Create the Decision Tree classifier
dt = DecisionTreeClassifier(random_state=0)

# Parameter grid for GridSearchCV
dt_param_grid = {
    "max_depth": dt_max_depth,
    "min_samples_split": dt_min_samples_split,
    "min_samples_leaf": dt_min_samples_leaf
}

In [12]:
# Ada Boost
# parameters you may consider
ada_n_estimators = [50, 100, 150]
ada_learning_rates = [0.1, 0.2, 0.3, 0.5]

param_grid_ada = {
    'n_estimators': ada_n_estimators,
    'learning_rate': ada_learning_rates
}

ada = AdaBoostClassifier(random_state=0)

In [13]:
# Gradient Boost
# parameters you may consider
gb_max_depth = [1, 3, 5, 7]
gb_n_estimators = [50, 100, 150]
gb_learning_rates = [0.1, 0.2, 0.3, 0.5]

param_grid_gb = {
    'max_depth': gb_max_depth,
    'n_estimators': gb_n_estimators,
    'learning_rate': gb_learning_rates
}


gb = GradientBoostingClassifier(random_state=0)

In [14]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to ‘sqrt’.
# parameters you may consider
rf_n_estimators = [10, 30, 60, 100]
rf_max_leaf_nodes = [6, 12]

param_grid_rf = {
    'n_estimators': rf_n_estimators,
    'max_leaf_nodes': rf_max_leaf_nodes
}

rf = RandomForestClassifier(
    criterion='entropy',
    max_features='sqrt',
    random_state=0
)

In [15]:
# SVM
# parameters you may consider
svm_c_values = [0.01, 0.1, 1, 5]
svm_gamma_values = [0.01, 0.1, 1, 10]

param_grid_svm = {
    'C': svm_c_values,
    'gamma': svm_gamma_values
}

svm = SVC(random_state=0)

### Part 2: Results

In [16]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn).
# The stratified folds from cvKFold should be provided to GridSearchCV.

# Split the pre-processed dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=0
)

# -------------------------
# KNN
# -------------------------

knn_grid = GridSearchCV(
    estimator=knn,
    param_grid=knn_param_grid,
    cv=cvKFold,
    scoring="accuracy"
)

# Perform parameter tuning using the training set
knn_grid.fit(X_train, y_train)

# Predict the test set using the best KNN model
knn_test_predictions = knn_grid.predict(X_test)

# Calculate test accuracy
knn_test_accuracy = accuracy_score(
    y_test,
    knn_test_predictions
)

# -------------------------
# Decision Tree
# -------------------------

dt_grid = GridSearchCV(
    estimator=dt,
    param_grid=dt_param_grid,
    cv=cvKFold,
    scoring="accuracy"
)

# Perform parameter tuning using the training set
dt_grid.fit(X_train, y_train)

# Predict the test set using the best Decision Tree model
dt_test_predictions = dt_grid.predict(X_test)

# Calculate test accuracy
dt_test_accuracy = accuracy_score(
    y_test,
    dt_test_predictions
)

# -------------------------
# KNN Results
# -------------------------

print("KNN best k:", knn_grid.best_params_["n_neighbors"])
print("KNN best p:", knn_grid.best_params_["p"])
print(
    "KNN cross-validation accuracy: {:.4f}".format(
        knn_grid.best_score_
    )
)
print(
    "KNN test set accuracy: {:.4f}".format(
        knn_test_accuracy
    )
)


# -------------------------
# Decision Tree Results
# -------------------------

print(
    "DT best max_depth:",
    dt_grid.best_params_["max_depth"]
)
print(
    "DT best min_samples_split:",
    dt_grid.best_params_["min_samples_split"]
)
print(
    "DT best min_samples_leaf:",
    dt_grid.best_params_["min_samples_leaf"]
)
print(
    "DT cross-validation accuracy: {:.4f}".format(
        dt_grid.best_score_
    )
)
print(
    "DT test set accuracy: {:.4f}".format(
        dt_test_accuracy
    )
)

# AdaBoost
ada_grid = GridSearchCV(
    estimator=ada,
    param_grid=param_grid_ada,
    cv=cvKFold,
    scoring="accuracy"
)
ada_grid.fit(X_train, y_train)
ada_test_predictions = ada_grid.predict(X_test)
ada_test_accuracy = accuracy_score(y_test, ada_test_predictions)

print("\n--- Ada Boost Results ---")
print(f"AdaBoost best n_estimators: {ada_grid.best_params_['n_estimators']}")
print(f"AdaBoost best learning_rate: {ada_grid.best_params_['learning_rate']}")
print(f"AdaBoost cross-validation accuracy: {ada_grid.best_score_:.4f}")
print(f"AdaBoost test set accuracy: {ada_test_accuracy:.4f}")

# Gradient Boosting
gb_grid = GridSearchCV(
    estimator=gb,
    param_grid=param_grid_gb,
    cv=cvKFold,
    scoring="accuracy"
)
gb_grid.fit(X_train, y_train)
gb_test_predictions = gb_grid.predict(X_test)
gb_test_accuracy = accuracy_score(y_test, gb_test_predictions)

print("\n--- Gradient Boost Results ---")
print(f"GB best max_depth: {gb_grid.best_params_['max_depth']}")
print(f"GB best n_estimators: {gb_grid.best_params_['n_estimators']}")
print(f"GB best learning_rate: {gb_grid.best_params_['learning_rate']}")
print(f"GB cross-validation accuracy: {gb_grid.best_score_:.4f}")
print(f"GB test set accuracy: {gb_test_accuracy:.4f}")

# SVM
svm_grid = GridSearchCV(
    estimator=svm,
    param_grid=param_grid_svm,
    cv=cvKFold,
    scoring="accuracy"
)
svm_grid.fit(X_train, y_train)
svm_test_predictions = svm_grid.predict(X_test)

print("\n--- SVM Results ---")
print(f"SVM best C: {svm_grid.best_params_['C']}")
print(f"SVM best gamma: {svm_grid.best_params_['gamma']}")
print(f"SVM cross-validation accuracy: {svm_grid.best_score_:.4f}")
print(f"SVM test set accuracy: {accuracy_score(y_test, svm_test_predictions):.4f}")


# Random Forest
rf_grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=cvKFold,
    scoring="accuracy"
)
rf_grid.fit(X_train, y_train)
rf_test_predictions = rf_grid.predict(X_test)

print("\n--- Random Forest Results ---")
print(f"RF best n_estimators: {rf_grid.best_params_['n_estimators']}")
print(f"RF best max_leaf_nodes: {rf_grid.best_params_['max_leaf_nodes']}")
print(f"RF cross-validation accuracy: {rf_grid.best_score_:.4f}")
print(f"RF test set accuracy: {accuracy_score(y_test, rf_test_predictions):.4f}")
print(f"RF test set macro average F1: {f1_score(y_test, rf_test_predictions, average='macro'):.4f}")
print(f"RF test set weighted average F1: {f1_score(y_test, rf_test_predictions, average='weighted'):.4f}")

KNN best k: 5
KNN best p: 1
KNN cross-validation accuracy: 0.9357
KNN test set accuracy: 0.9333
DT best max_depth: 5
DT best min_samples_split: 2
DT best min_samples_leaf: 2
DT cross-validation accuracy: 0.9347
DT test set accuracy: 0.9286



--- Ada Boost Results ---
AdaBoost best n_estimators: 50
AdaBoost best learning_rate: 0.1
AdaBoost cross-validation accuracy: 0.9408
AdaBoost test set accuracy: 0.9429



--- Gradient Boost Results ---
GB best max_depth: 1
GB best n_estimators: 50
GB best learning_rate: 0.1
GB cross-validation accuracy: 0.9418
GB test set accuracy: 0.9429



--- SVM Results ---
SVM best C: 1
SVM best gamma: 1
SVM cross-validation accuracy: 0.9408
SVM test set accuracy: 0.9429



--- Random Forest Results ---
RF best n_estimators: 10
RF best max_leaf_nodes: 6
RF cross-validation accuracy: 0.9418
RF test set accuracy: 0.9310
RF test set macro average F1: 0.9293
RF test set weighted average F1: 0.9308


### Test your code

In [17]:
#load the test dataset to test out your model 


## **3. Reflection and Discussion**



Overall, all eight classifiers performed well on the rice dataset. In Part 1, Logistic Regression achieved a cross-validation accuracy of 0.9386, which was higher than Naive Bayes at 0.9264. Naive Bayes may perform worse because the rice features are not completely independent.

For the tuned models, Gradient Boosting and Random Forest had the highest cross-validation accuracy of 0.9418. AdaBoost and SVM were close behind at 0.9408. On the test set, AdaBoost, Gradient Boosting and SVM achieved the best accuracy of 0.9429. KNN achieved 0.9333, Decision Tree achieved 0.9286, and Random Forest achieved 0.9310. The macro and weighted F1 scores of Random Forest were 0.9293 and 0.9308, so its performance was similar across both classes.

Hyperparameter tuning helped select suitable model complexity. KNN selected `k=5` and `p=1`, while the Decision Tree selected a maximum depth of 5. Both boosting models selected a low learning rate of 0.1, and Gradient Boosting selected shallow trees. These choices show that more complex models do not always give better results. GridSearchCV also selected parameters using only the training data, so the test set remained separate from the tuning process. However, we cannot calculate the exact improvement from tuning because default Part 2 models were not tested.

Each classifier has different advantages and disadvantages. Logistic Regression and Naive Bayes are fast and simple, but they make stronger assumptions about the data. KNN can model nonlinear patterns but is sensitive to scaling and can be slow during prediction. Decision Trees are easy to understand but can overfit. Random Forest is more stable than a single tree but is less interpretable. AdaBoost and Gradient Boosting performed very well, although they take longer to train and depend on their parameters. SVM also performed well, but it is sensitive to feature scaling and the values of `C` and `gamma`. Based on the results, Gradient Boosting, AdaBoost and SVM gave the best overall performance.

## **AI Acknowledgement**

OpenAI Codex was used to assist with reviewing the notebook against the marking rubric, resolving Git merge conflicts, consolidating the classifier implementations, improving code readability. The group reviewed the resulting code and discussion and verified the notebook by executing it on both the supplied rice dataset and `test-before.csv`.